In [ ]:
!pip install torch torchvision timm opencv-python matplotlib scikit-learn tqdm

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
from tqdm import tqdm

#Training Datasets

In [ ]:
!unzip DukeMTMC_dataset.zip -d /dukemtmc
!unzip market_1501_dataset.zip -d /market1501


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32
EPOCHS = 50
LR = 3e-4

NUM_PARTS = 6   #horizontal splitting

# Individual dataset paths
MARKET_PATH = "/market1501"
DUKE_PATH = "/dukemtmc"

# Combined dataset root (merge loaders, not folders)
DATASET_PATHS = {
    "market": MARKET_PATH,
    "duke": DUKE_PATH
}

# Number of classes (will be computed dynamically after loading datasets)
NUM_CLASSES = None

In [ ]:
VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png')

class ReIDDataset(Dataset):
   def __len__(self):
          return len(self.samples)

   def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label
   def __init__(self, roots, transform=None):
        self.samples = []
        self.transform = transform
        self.pids = []  # added this for PK sampler

        current_label_offset = 0

        for name, root in roots.items():
            img_dir = os.path.join(root, "bounding_box_train")
            pid_container = set()

            # collect IDs
            for img_name in os.listdir(img_dir):
                if not img_name.lower().endswith(VALID_EXTENSIONS):
                    continue   #skip non-image files

                pid = int(img_name.split('_')[0])
                if pid == -1:
                    continue
                pid_container.add(pid)

            pid2label = {
                pid: idx + current_label_offset
                for idx, pid in enumerate(pid_container)
            }

            # store samples
            for img_name in os.listdir(img_dir):
                if not img_name.lower().endswith(VALID_EXTENSIONS):
                    continue   #skip non-image files

                pid = int(img_name.split('_')[0])
                if pid == -1:
                    continue

                label = pid2label[pid]

                self.samples.append((
                    os.path.join(img_dir, img_name),
                    label
                ))
                self.pids.append(label)  # original PID for PK sampler

            current_label_offset += len(pid_container)

        self.num_classes = current_label_offset

#Training Transformations

In [ ]:
train_tfms = T.Compose([
    T.Resize((256, 128)),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop((256, 128)),
    T.ColorJitter(0.15, 0.15, 0.15, 0.05),

    T.ToTensor(),

    # ImageNet normalization (better for ResNet)
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),

    # important for occlusion robustness
    T.RandomErasing(p=0.5, scale=(0.02, 0.4), ratio=(0.3, 3.3))
])

#PK Sampler

In [ ]:
from torch.utils.data import Sampler
import random

class RandomIdentitySampler(Sampler):
    # PK sampling for person Re-ID.
    def __init__(self, data_source, num_identities, num_instances):
        self.data_source = data_source
        self.num_identities = num_identities  # P
        self.num_instances = num_instances    # K

        # create a dict {pid: [indices]}
        self.index_dic = {}
        for idx, pid in enumerate(data_source.pids):
            self.index_dic.setdefault(pid, []).append(idx)
        self.pids = list(self.index_dic.keys())

    def __iter__(self):
        batch = []
        while len(batch) < len(self.data_source):
            pid_batch = random.sample(self.pids, self.num_identities)
            for pid in pid_batch:
                idxs = self.index_dic[pid]
                if len(idxs) < self.num_instances:
                    idxs = random.choices(idxs, k=self.num_instances)
                else:
                    idxs = random.sample(idxs, self.num_instances)
                batch.extend(idxs)
        return iter(batch[:len(self.data_source)])

    def __len__(self):
        return len(self.data_source)

In [ ]:
P = 8   # number of identities per batch
K = 4   # number of images per identity

train_dataset = ReIDDataset(
    roots={
        "market": "/market1501",
        "duke": "/dukemtmc"
    },
    transform=train_tfms
)

train_loader = DataLoader(
    train_dataset,
    batch_size=P*K,
    sampler=RandomIdentitySampler(train_dataset, num_identities=P, num_instances=K),
    num_workers=2,
    pin_memory=True
)

NUM_CLASSES = train_dataset.num_classes
print("Classes:", NUM_CLASSES)

Classes: 1453


#Part-Attention Module

In [ ]:
class PartAttention(nn.Module):
    def __init__(self, in_channels, num_parts=6):
        super().__init__()
        self.num_parts = num_parts

        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 4, 1),
            nn.ReLU(),
            nn.Conv2d(in_channels // 4, num_parts, 1),  # output = parts
            nn.Sigmoid()
        )

    def forward(self, x):
        B, C, H, W = x.size()

        # Generate attention maps for each part
        attn_maps = self.attn(x)   # [B, num_parts, H, W]

        part_features = []

        # Split feature map horizontally
        stripe_h = H // self.num_parts

        for i in range(self.num_parts):
            part = x[:, :, i*stripe_h:(i+1)*stripe_h, :]
            attn = attn_maps[:, i:i+1, i*stripe_h:(i+1)*stripe_h, :]

            weighted_part = part * attn
            pooled = weighted_part.mean(dim=[2, 3])  # GAP

            part_features.append(pooled)

        # Concatenate all part features
        part_features = torch.cat(part_features, dim=1)  # [B, C*num_parts]

        # L2 normalization here
        part_features = F.normalize(part_features, p=2, dim=1)

        return part_features

#model

In [ ]:
class ReIDModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.backbone = timm.create_model(
            "resnet50",
            pretrained=True,
            features_only=True
        )

        self.out_channels = self.backbone.feature_info[-1]['num_chs']

        # single attention module (shared)
        self.part_attn = PartAttention(self.out_channels, NUM_PARTS)

        self.bnneck = nn.BatchNorm1d(self.out_channels * NUM_PARTS)

        self.classifier = nn.Linear(self.out_channels * NUM_PARTS, num_classes)

    # initialize classifier weights for stable CE training
        nn.init.kaiming_normal_(self.classifier.weight, mode='fan_out')
        nn.init.constant_(self.classifier.bias, 0)

    def forward(self, x):
        feat = self.backbone(x)[-1]   # [B, C, H, W]

        # get part-based embedding directly
        embedding = self.part_attn(feat)   # [B, C * NUM_PARTS]

        # BNNeck
        feat_bn = self.bnneck(embedding)

        logits = self.classifier(feat_bn)

        return embedding, logits

#Loss Functions

In [ ]:
# CE with label smoothing
ce_loss = nn.CrossEntropyLoss(label_smoothing=0.1)

def batch_hard_triplet_loss(embeddings, labels):

    # pairwise distance matrix
    dist = torch.cdist(embeddings, embeddings)

    # positive mask
    mask_pos = labels.unsqueeze(1) == labels.unsqueeze(0)

    # negative mask
    mask_neg = ~mask_pos

    # remove anchor
    mask_pos.fill_diagonal_(False)

    # hardest positive
    hardest_pos = (dist * mask_pos.float()).max(dim=1)[0]

    # mask positives for negative search
    max_dist = dist.max().detach()
    dist_neg = dist + max_dist * mask_pos.float()

    # hardest negative
    hardest_neg = dist_neg.min(dim=1)[0]

    # soft margin triplet
    loss = torch.log1p(torch.exp(hardest_pos - hardest_neg))

    return loss.mean()

#Optimiser & Step scheduler

In [ ]:
model = ReIDModel(NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=5e-4)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=20,
    gamma=0.1
)

#Training

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader)

    for imgs, labels in loop:
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        embeddings, logits = model(imgs)

         # normalize embeddings for triplet loss
        embeddings = F.normalize(embeddings, p=2, dim=1)

        # classification loss on BNNeck
        loss_ce = ce_loss(logits, labels)

        # batch-hard triplet loss
        loss_tri = batch_hard_triplet_loss(embeddings, labels)

        # combined loss (weighting can be adjusted)
        loss = loss_ce + loss_tri

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10)
        optimizer.step()

        total_loss += loss.item()

        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(
            loss=loss.item(),
            ce=loss_ce.item(),
            tri=loss_tri.item()
        )

    scheduler.step()
    torch.save(model.state_dict(), "best_model.pth")
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

#Test dataset loader

In [ ]:
# market1501 and dukemtmc

class ReIDTestDataset(Dataset):
    def __init__(self, root, mode="query", transform=None):
        self.samples = []
        self.transform = transform

        if mode == "query":
            img_dir = os.path.join(root, "query")
        else:
            img_dir = os.path.join(root, "bounding_box_test")

        for img_name in os.listdir(img_dir):
            if not img_name.lower().endswith(('.jpg', '.png')):
                continue

            pid = int(img_name.split('_')[0])
            camid = int(img_name.split('_')[1][1])

            self.samples.append((
                os.path.join(img_dir, img_name),
                pid,
                camid
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, pid, camid = self.samples[idx]
        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, pid, camid

In [ ]:
#occluded Duke

class OccludedReIDDataset:
    def __init__(self, root, list_file, transform):
        self.root = root
        self.transform = transform

        self.img_paths = []
        self.pids = []
        self.camids = []

        # Correct folder mapping
        if list_file == "query.list":
            subdir = "query"
        elif list_file == "gallery.list":
            subdir = "bounding_box_test"
        else:
            raise ValueError("Unknown list file")

        with open(os.path.join(root, list_file)) as f:
            for line in f:
                fname = line.strip()

                img_path = os.path.join(root, subdir, fname)

                pid = int(fname.split('_')[0])
                camid = int(fname.split('_')[1][1]) - 1

                self.img_paths.append(img_path)
                self.pids.append(pid)
                self.camids.append(camid)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]

        # Debug check
        if not os.path.exists(img_path):
            print("Missing:", img_path)

        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, self.pids[idx], self.camids[idx]

    def __len__(self):
        return len(self.img_paths)

In [ ]:
# Occluded ReID

import os
from torch.utils.data import Dataset
from PIL import Image


class AdvancedReIDDataset(Dataset):
    def __init__(self, root, mode="query", transform=None, dataset_type="occluded"):

        self.transform = transform
        self.samples = []

        self.root = os.path.join(root, mode)
        self._process_files()

    def _process_files(self):
        img_names = os.listdir(self.root)

        for img_name in img_names:
            if not img_name.lower().endswith((".jpg", ".png")):
                continue

            img_path = os.path.join(self.root, img_name)

            # Extract PID
            # example: 0001_2.jpg → pid=1
            pid = int(img_name.split('_')[0])

            # Extract CAMID (if exists)
            if '_' in img_name:
                try:
                    camid = int(img_name.split('_')[1].split('.')[0])
                except:
                    camid = 0
            else:
                camid = 0

            self.samples.append((img_path, pid, camid))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, pid, camid = self.samples[idx]

        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, pid, camid

#occluded datasets for Evaluation

In [ ]:
!unzip Occluded_Duke_dataset.zip -d /occluded_duke
!unzip OccludedREID.zip -d /occluded_reid

#Feature Extraction

In [ ]:
import torch
import torch.nn.functional as F

def extract_features(model, loader):
    model.eval()

    features, pids, camids = [], [], []

    with torch.no_grad():
        for imgs, pid, camid in loader:
            imgs = imgs.to(DEVICE)

            embedding, _ = model(imgs)
            embedding = F.normalize(embedding, dim=1)

            features.append(embedding.cpu())
            pids.extend(pid)
            camids.extend(camid)

    return torch.cat(features), pids, camids

#Reranking

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist

def re_ranking(qf, gf, k1=20, k2=6, lambda_value=0.3):

    query_num = qf.shape[0]
    gallery_num = gf.shape[0]
    all_num = query_num + gallery_num

    feat = np.vstack((qf, gf)).astype(np.float32)

    # Step 1: Compute distance
    original_dist = cdist(feat, feat)
    original_dist = original_dist / (np.max(original_dist) + 1e-12)

    V = np.zeros_like(original_dist, dtype=np.float32)

    initial_rank = np.argsort(original_dist, axis=1)

    # Step 2: k-reciprocal neighbors
    for i in range(all_num):

        forward_k = initial_rank[i, :k1+1]
        backward_k = initial_rank[forward_k, :k1+1]

        fi = np.where(backward_k == i)[0]
        k_reciprocal = forward_k[fi]

        weight = np.exp(-original_dist[i, k_reciprocal])
        V[i, k_reciprocal] = weight / np.sum(weight)

    # Step 3: Query-Gallery split
    original_dist = original_dist[:query_num, query_num:]
    V_q = V[:query_num]
    V_g = V[query_num:]

    # Step 4: Jaccard distance
    jaccard_dist = 1 - np.dot(V_q, V_g.T)

    final_dist = jaccard_dist * (1 - lambda_value) + original_dist * lambda_value

    return final_dist

#Distance

In [ ]:
import numpy as np

def evaluate(query_feat, query_pids, query_camids,
             gallery_feat, gallery_pids, gallery_camids):

    query_feat = F.normalize(query_feat, dim=1)
    gallery_feat = F.normalize(gallery_feat, dim=1)

    distmat = re_ranking(
    query_feat.cpu().numpy(),
    gallery_feat.cpu().numpy()
    )

    # distmat = torch.cdist(query_feat, gallery_feat).numpy()      #For evaluation without re-ranking

    num_q = distmat.shape[0]
    all_cmc = []
    all_AP = []

    for i in range(num_q):
        q_pid = query_pids[i]
        q_cam = query_camids[i]

        order = np.argsort(distmat[i])

        remove = (gallery_pids == q_pid) & (gallery_camids == q_cam)
        keep = np.invert(remove)

        matches = (gallery_pids == q_pid).astype(int)

        matches = matches[order][keep]

        if not np.any(matches):
            continue

        # CMC
        cmc = matches.cumsum()
        cmc[cmc > 1] = 1
        all_cmc.append(cmc[:50])

        # AP
        num_rel = matches.sum()
        tmp_cmc = matches.cumsum()
        precision = tmp_cmc / (np.arange(len(matches)) + 1)
        AP = (precision * matches).sum() / num_rel
        all_AP.append(AP)

    cmc = np.mean(all_cmc, axis=0)
    mAP = np.mean(all_AP)

    return cmc, mAP

#test transformations

In [ ]:
test_tfms = T.Compose([
    T.Resize((256, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

#Evaluation

In [ ]:
#market & duke

def run_eval(model, root):
    query_set = ReIDTestDataset(root, "query", test_tfms)
    gallery_set = ReIDTestDataset(root, "gallery", test_tfms)

    query_loader = DataLoader(query_set, batch_size=64, shuffle=False)
    gallery_loader = DataLoader(gallery_set, batch_size=64, shuffle=False)

    qf, qp, qc = extract_features(model, query_loader)
    gf, gp, gc = extract_features(model, gallery_loader)

    #apply slicing or not for re-ranking
    # gf = gf[:7000]
    # gp = gp[:7000]
    # gc = gc[:7000]

    cmc, mAP = evaluate(qf, np.array(qp), np.array(qc),
                        gf, np.array(gp), np.array(gc))

    print(f"Rank-1: {cmc[0]:.4f}, mAP: {mAP:.4f}")

In [ ]:
#occluded duke

def run_occluded_eval(model, root):
    query_set = OccludedReIDDataset(root, "query.list", test_tfms)
    gallery_set = OccludedReIDDataset(root, "gallery.list", test_tfms)

    query_loader = DataLoader(query_set, batch_size=64, shuffle=False)
    gallery_loader = DataLoader(gallery_set, batch_size=64, shuffle=False)

    qf, qp, qc = extract_features(model, query_loader)
    gf, gp, gc = extract_features(model, gallery_loader)

    #apply slicing or not for re-ranking
    # gf = gf[:7000]
    # gp = gp[:7000]
    # gc = gc[:7000]

    cmc, mAP = evaluate(qf, np.array(qp), np.array(qc),
                        gf, np.array(gp), np.array(gc))

    print(f"[Occluded] Rank-1: {cmc[0]:.4f}, mAP: {mAP:.4f}")

In [ ]:
#Occluded reid

def run_new_eval(model, root, dataset_type="occluded"):

    query_set = AdvancedReIDDataset(
        root,
        mode="query",
        transform=test_tfms,
        dataset_type=dataset_type
    )

    gallery_set = AdvancedReIDDataset(
        root,
        mode="gallery",
        transform=test_tfms,
        dataset_type=dataset_type
    )

    query_loader = DataLoader(query_set, batch_size=64, shuffle=False)
    gallery_loader = DataLoader(gallery_set, batch_size=64, shuffle=False)

    qf, qp, qc = extract_features(model, query_loader)
    gf, gp, gc = extract_features(model, gallery_loader)

    cmc, mAP = evaluate(qf, np.array(qp), np.array(qc),
                        gf, np.array(gp), np.array(gc))

    print(f"Rank-1: {cmc[0]:.4f}, mAP: {mAP:.4f}")

#Model

In [ ]:
model = ReIDModel(num_classes=NUM_CLASSES)
model.load_state_dict(torch.load("best_model.pth"))
model.cuda()
model.eval()

#Rank-1 & mAP

In [ ]:
print("Market1501:")
run_eval(model, "/market1501")

In [ ]:
print("DukeMTMC:")
run_eval(model, "/dukemtmc")

In [ ]:
print("Occluded Duke:")
run_occluded_eval(model ,"/occluded_duke")

In [ ]:
print("Occluded Reid:")
run_new_eval(model ,"/occluded_reid/Occluded_REID")